# 02 · ETL Failure Lab — לתקן את מה שנכשל
**Goal:** לממש עיבוד חוזר ללא כפילויות, לזהות רשומה שגויה ולבצע replay.  
**Setup:** Python + SQLite בלבד. הנתונים סינתטיים.  
**CS bridge:** uniqueness, state, idempotency, transaction semantics.  
**תזכורת:** בדיוק-once end-to-end היא טענה חזקה שתלויה בכל המערכות; הניסוי כאן מלמד idempotent sink בלבד.

## צפה ולמד — API ingestion with dlt (2026)

[**פתח את הסרטון המקורי ב־YouTube**](https://www.youtube.com/watch?v=5eMytPBgmVs) · מקטע מומלץ: 5–15 דקות לנושא שבו זיהית פער.

[![Video thumbnail](https://img.youtube.com/vi/5eMytPBgmVs/hqdefault.jpg)](https://www.youtube.com/watch?v=5eMytPBgmVs)

ב־Jupyter/VS Code הנגן בתא הבא עשוי לפעול אם יש רשת; ב־GitHub preview הנגן אינו מובטח. תוכל להמשיך בתרגיל גם בלי וידאו.

In [1]:
# Optional video player (does not download or store the video)
try:
    from IPython.display import display, YouTubeVideo
    display(YouTubeVideo("5eMytPBgmVs", width=720, height=405))
except ImportError:
    print("Open the YouTube link in the Markdown above.")

**צפייה משלימה:** [DataTalks.Club workshop: ingestion & incremental loading](https://github.com/DataTalksClub/data-engineering-zoomcamp/tree/main/cohorts/2026/workshops). פתח את הקישור לפני/אחרי התרגול, לא במקום כתיבת הקוד.

## 1. חידת פתיחה
Pipeline מצליח לטעון שלוש רשומות אך נופל רגע לפני שהוא מסמן שהצליח. Scheduler מנסה שוב. האם נקבל 6 רשומות?

In [2]:
import sqlite3
incoming=[
    {"event_id":"e1","magnitude":3.1},
    {"event_id":"e2","magnitude":4.2},
    {"event_id":"e3","magnitude":2.4},
    {"event_id":"bad","magnitude":None},
]
conn=sqlite3.connect(":memory:")
conn.execute("CREATE TABLE events(event_id TEXT PRIMARY KEY,magnitude REAL NOT NULL)")

## 2. Validate → Quarantine → Upsert
כלל ה־validation כאן הוא לצורך דוגמה: מזהה nonempty, magnitude מספרי. תעדו את הסיבה לדחייה במקום לזרוק אותה בלי מידע.

In [3]:
def ingest(batch):
    bad=[]; accepted=[]
    for r in batch:
        if not r.get("event_id") or not isinstance(r.get("magnitude"),(int,float)):
            bad.append({"record":r,"reason":"invalid event_id or magnitude"})
        else:
            accepted.append((r["event_id"],r["magnitude"]))
    with conn:
        conn.executemany("""INSERT INTO events(event_id,magnitude) VALUES (?,?)
            ON CONFLICT(event_id) DO UPDATE SET magnitude=excluded.magnitude""",accepted)
    return {"accepted":len(accepted),"rejected":bad}
first=ingest(incoming)
rows_after_first=conn.execute("SELECT COUNT(*) FROM events").fetchone()[0]
second=ingest(incoming)
rows_after_replay=conn.execute("SELECT COUNT(*) FROM events").fetchone()[0]
print(first,"count after 1st:",rows_after_first,"after retry:",rows_after_replay)
assert rows_after_first == rows_after_replay == 3
assert len(first["rejected"]) == 1

{'accepted': 3, 'rejected': [{'record': {'event_id': 'bad', 'magnitude': None}, 'reason': 'invalid event_id or magnitude'}]} count after 1st: 3 after retry: 3


## 3. late update ו־versioning
דמיין ש־e2 מגיע שוב עם magnitude 4.0. הפתרון הפשוט עושה overwrite. האם צריך גם לשמור versions, `source_updated_at` ו־audit trail? זה תלוי בחוזה הנתונים.

In [4]:
ingest([{"event_id":"e2","magnitude":4.0}])
observed=conn.execute("SELECT magnitude FROM events WHERE event_id='e2'").fetchone()[0]
print("latest stored magnitude:",observed)
assert observed == 4.0

latest stored magnitude: 4.0


## 4. מבחן העברה
- מה יקרה אם שני עדכונים ל־e2 מגיעים בסדר הפוך? כתוב הגנת `source_updated_at`.
- איך מבצעים transaction אם batch כולל 100,000 רשומות?
- כמה פעמים מותר לנסות שוב אחרי 429? איך מזהים poison pill?
- מדוע `ON CONFLICT` לא פותר אוטומטית **כל** בעיות exactly-once?

**Side Quest:** בנה טבלת `ingestion_runs` ורשום לכל run started_at, finished_at, rejected_count.  
**Next:** [Stage 03](../curriculum/modules/03.md) ומסלול [Supabase](../curriculum/modules/04.md).